In [ ]:
%pip install jieba gensim flask
import os
import sys
import json
import threading
import webbrowser
import time
import sqlite3
import re
from pathlib import Path
from flask import Flask, request, jsonify, send_file, Response
from gensim.models import Word2Vec
import numpy as np
import jieba 

app = Flask(__name__)

if getattr(sys, 'frozen', False):
    BASE_DIR = Path(sys.executable).parent
else:
    BASE_DIR = Path(__file__).parent

MODEL_PATH  = BASE_DIR/"imessage_word2vec.model"
VECTORS_TSV = BASE_DIR / "vectors.tsv"
METADATA_TSV = BASE_DIR / "metadata.tsv"
INDEX_HTML = BASE_DIR / "index.html"
TOP_N = 10_000
PORT = 5050

def train_from_messages():
    db_path = os.path.expanduser("~/Library/Messages/chat.db")
    if not os.path.exists(db_path):
        print("Cannot find iMessage DB")
        return(None)
    print("Reading messages...")
    conn =sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT text FROM message WHERE text IS NOT NULL")
    texts = [row[0] for row in cursor.fetchall()]
    conn.close()
    print(f" {len(texts)} messages")

    def clean(text):
        text = re.sub(r"\s", "", text.strip()).lower()
        tokens = jieba.lcut(text)
        return[t for t in tokens if t.strip()]
    
    sentences = [clean(t) for t in texts]
    sentences = [s for s in sentences if s]
    print("Training Word2Vec model...")
    m = Word2Vec(sentences = sentences, vector_size=100, window = 5, min_count = 5, workers=4)
    m.save(str(MODEL_PATH))
    print("Model saved {MODEL_PATH} ")
    print(f"Ready. Vocabulary: {len(m.wv.key_to_index)} words")
    return m

#load model
if MODEL_PATH.exists():
    print(f"Loading model from {MODEL_PATH}")
    model  = Word2Vec.load(str(MODEL_PATH))
else:
    print("Training model")
    model = train_from_messages()
print(f"Ready, Vocabulary: {len(model.wv.key_to_index)} words")
#TSV files


@app.route("/vectors.tsv")
def vectors():
    return send_file(VECTORS_TSV, mimetype="text/tab-separated-values")
@app.route("/metadata.tsv")
def metadata():
    return send_file(METADATA_TSV, mimetype="text/tab-separated-values")
@app.route("/api/ping")
def ping():
    return jsonify({"status":"ok", "vocab_size":len(m.wv.key_to_index)})


